In [1]:
pip install yfinance pandas matplotlib seaborn openpyxl jupyter

  Using cached matplotlib-3.10.8-cp314-cp314-macosx_10_13_x86_64.whl.metadata (52 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached websockets-16.0-cp314-cp314-macosx_10_15_x86_64.whl.metadata (6.8 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (2.7 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.

In [2]:
import yfinance as yf
import pandas as pd
import time
import os
import warnings

warnings.filterwarnings("ignore")

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
banks = {
    "Barclays": "BARC.L",
    "BNP Paribas": "BNP.PA",
    "Deutsche Bank": "DBK.DE"
}

print("Banks selected for analysis:")
for name, ticker in banks.items():
    print(f"  {name} — Ticker: {ticker}")

Banks selected for analysis:
  Barclays — Ticker: BARC.L
  BNP Paribas — Ticker: BNP.PA
  Deutsche Bank — Ticker: DBK.DE


In [4]:
print("Pulling 3 years of daily closing prices...\n")

price_data = {}

for name, ticker in banks.items():
    stock = yf.Ticker(ticker)
    hist = stock.history(period="3y")
    
    if hist.empty:
        print(f"WARNING — No price data returned for {name}. Check ticker symbol.")
    else:
        price_data[name] = hist["Close"]
        print(f"{name}: {len(hist)} trading days pulled successfully")
    
    time.sleep(1)

prices_df = pd.DataFrame(price_data)

print(f"\nPrice DataFrame shape: {prices_df.shape}")
print("\nMost recent 3 rows:")
print(prices_df.tail(3))

Pulling 3 years of daily closing prices...

Barclays: 759 trading days pulled successfully
BNP Paribas: 767 trading days pulled successfully
Deutsche Bank: 763 trading days pulled successfully

Price DataFrame shape: (1526, 3)

Most recent 3 rows:
                             Barclays  BNP Paribas  Deutsche Bank
Date                                                             
2026-03-19 00:00:00+00:00  381.600006          NaN            NaN
2026-03-19 23:00:00+00:00         NaN    82.129997          24.76
2026-03-20 00:00:00+00:00  373.899994          NaN            NaN


In [5]:
print("Pulling financial metrics snapshot...\n")

metrics = {}

for name, ticker in banks.items():
    stock = yf.Ticker(ticker)
    info = stock.info
    
    metrics[name] = {
        "Current Price":      info.get("currentPrice"),
        "Currency":           info.get("currency"),
        "Market Cap":         info.get("marketCap"),
        "P/B Ratio":          info.get("priceToBook"),
        "P/E Ratio":          info.get("trailingPE"),
        "ROE":                info.get("returnOnEquity"),
        "ROA":                info.get("returnOnAssets"),
        "Profit Margin":      info.get("profitMargins"),
        "Total Revenue":      info.get("totalRevenue"),
        "Total Assets":       info.get("totalAssets"),
        "Book Value":         info.get("bookValue"),
    }
    
    print(f"{name}: metrics pulled")
    time.sleep(1)

metrics_df = pd.DataFrame(metrics).T

print("\nMetrics snapshot:")
print(metrics_df)

Pulling financial metrics snapshot...

Barclays: metrics pulled
BNP Paribas: metrics pulled
Deutsche Bank: metrics pulled

Metrics snapshot:
              Current Price Currency   Market Cap P/B Ratio P/E Ratio  \
Barclays              373.9      GBp  51501977600  79.58706  8.902381   
BNP Paribas           82.13      EUR  91721531392  0.786354  7.981535   
Deutsche Bank         24.76      EUR  48023834624  0.583659  8.012945   

                   ROE      ROA Profit Margin Total Revenue Total Assets  \
Barclays       0.09572  0.00471       0.26743   26818000896         None   
BNP Paribas    0.09656  0.00468       0.25502   47936999424         None   
Deutsche Bank  0.08302  0.00481       0.22222   29727000576         None   

              Book Value  
Barclays           4.698  
BNP Paribas      104.444  
Deutsche Bank     42.422  


In [6]:
print("Running data quality check...\n")

missing = metrics_df.isnull().sum()

print("Missing values per metric:")
print(missing)

total_missing = missing.sum()
print(f"\nTotal missing values: {total_missing}")

if total_missing == 0:
    print("\nData quality check PASSED — no missing values")
else:
    print("\nWARNING — missing values detected. Review before proceeding to analysis.")
    print("\nAffected fields:")
    print(metrics_df[metrics_df.isnull().any(axis=1)])

Running data quality check...

Missing values per metric:
Current Price    0
Currency         0
Market Cap       0
P/B Ratio        0
P/E Ratio        0
ROE              0
ROA              0
Profit Margin    0
Total Revenue    0
Total Assets     3
Book Value       0
dtype: int64

Total missing values: 3

WARNING — missing values detected. Review before proceeding to analysis.

Affected fields:
              Current Price Currency   Market Cap P/B Ratio P/E Ratio  \
Barclays              373.9      GBp  51501977600  79.58706  8.902381   
BNP Paribas           82.13      EUR  91721531392  0.786354  7.981535   
Deutsche Bank         24.76      EUR  48023834624  0.583659  8.012945   

                   ROE      ROA Profit Margin Total Revenue Total Assets  \
Barclays       0.09572  0.00471       0.26743   26818000896         None   
BNP Paribas    0.09656  0.00468       0.25502   47936999424         None   
Deutsche Bank  0.08302  0.00481       0.22222   29727000576         None   

     

In [7]:
print("Pulling annual income statements...\n")

income_statements = {}

for name, ticker in banks.items():
    stock = yf.Ticker(ticker)
    income = stock.financials
    
    if income.empty:
        print(f"WARNING — No income statement data for {name}")
    else:
        income_statements[name] = income
        print(f"{name}: {income.shape[1]} years of data")
        print(f"  Available fields: {income.index.tolist()}\n")
    
    time.sleep(1)

Pulling annual income statements...

Barclays: 5 years of data
  Available fields: ['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Otherunder Preferred Stock Dividend', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Extraordinary', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Special Income Charges', 'Other Special Charges', 'Impairment Of Capital Assets', 'Restructuring And Mergern Acquisition', 'Gain On Sale Of Security', 'Operating Expense', 'Other Operating Expens

In [8]:
print("Pulling annual balance sheets...\n")

balance_sheets = {}

for name, ticker in banks.items():
    stock = yf.Ticker(ticker)
    balance = stock.balance_sheet
    
    if balance.empty:
        print(f"WARNING — No balance sheet data for {name}")
    else:
        balance_sheets[name] = balance
        print(f"{name}: {balance.shape[1]} years of data")
        print(f"  Available fields: {balance.index.tolist()}\n")
    
    time.sleep(1)

Pulling annual balance sheets...

Barclays: 4 years of data
  Available fields: ['Treasury Shares Number', 'Ordinary Shares Number', 'Share Issued', 'Total Debt', 'Tangible Book Value', 'Invested Capital', 'Net Tangible Assets', 'Common Stock Equity', 'Total Capitalization', 'Total Equity Gross Minority Interest', 'Minority Interest', 'Stockholders Equity', 'Other Equity Interest', 'Retained Earnings', 'Additional Paid In Capital', 'Capital Stock', 'Common Stock', 'Total Liabilities Net Minority Interest', 'Derivative Product Liabilities', 'Long Term Debt And Capital Lease Obligation', 'Payables', 'Other Payable', 'Total Tax Payable', 'Total Assets', 'Defined Pension Benefit', 'Investments And Advances', 'Investmentin Financial Assets', 'Available For Sale Securities', 'Financial Assets Designatedas Fair Value Through Profitor Loss Total', 'Trading Securities', 'Long Term Equity Investment', 'Investmentsin Joint Venturesat Cost', 'Investmentsin Associatesat Cost', 'Investment Propertie

In [10]:
print("Saving all data...\n")


prices_df.to_csv("../data/bank_prices.csv")
print("Saved: bank_prices.csv")

metrics_df.to_csv("../data/bank_metrics.csv")
print("Saved: bank_metrics.csv")

for name, df in income_statements.items():
    filename = f"../data/income_{name.replace(' ', '_')}.csv"
    df.to_csv(filename)
    print(f"Saved: income_{name.replace(' ', '_')}.csv")

for name, df in balance_sheets.items():
    filename = f"../data/balance_{name.replace(' ', '_')}.csv"
    df.to_csv(filename)
    print(f"Saved: balance_{name.replace(' ', '_')}.csv")

print(f"\nAll files saved. Contents of data folder:")
print(os.listdir("../data"))

Saving all data...

Saved: bank_prices.csv
Saved: bank_metrics.csv
Saved: income_Barclays.csv
Saved: income_BNP_Paribas.csv
Saved: income_Deutsche_Bank.csv
Saved: balance_Barclays.csv
Saved: balance_BNP_Paribas.csv
Saved: balance_Deutsche_Bank.csv

All files saved. Contents of data folder:
['balance_Barclays.csv', 'income_BNP_Paribas.csv', 'balance_Deutsche_Bank.csv', 'bank_metrics.csv', 'balance_BNP_Paribas.csv', 'income_Barclays.csv', 'income_Deutsche_Bank.csv', 'bank_prices.csv']


In [11]:
print("="*50)
print("NOTEBOOK 1 COMPLETE — DATA COLLECTION SUMMARY")
print("="*50)

print(f"\nBanks analysed: {list(banks.keys())}")
print(f"Price data: {prices_df.shape[0]} trading days x {prices_df.shape[1]} banks")
print(f"Metrics collected: {len(metrics_df.columns)} fields per bank")
print(f"Income statements: {len(income_statements)} banks")
print(f"Balance sheets: {len(balance_sheets)} banks")

print("\nNext step: Open 02_ratio_analysis.ipynb")

NOTEBOOK 1 COMPLETE — DATA COLLECTION SUMMARY

Banks analysed: ['Barclays', 'BNP Paribas', 'Deutsche Bank']
Price data: 1526 trading days x 3 banks
Metrics collected: 11 fields per bank
Income statements: 3 banks
Balance sheets: 3 banks

Next step: Open 02_ratio_analysis.ipynb


In [12]:
# Check exactly what fields came back in income statements
for name, df in income_statements.items():
    print(f"\n{name} income statement fields:")
    print(df.index.tolist())

# Check exactly what fields came back in balance sheets  
for name, df in balance_sheets.items():
    print(f"\n{name} balance sheet fields:")
    print(df.index.tolist())

# Check metrics table
print("\nMetrics table:")
print(metrics_df)


Barclays income statement fields:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Otherunder Preferred Stock Dividend', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Extraordinary', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Special Income Charges', 'Other Special Charges', 'Impairment Of Capital Assets', 'Restructuring And Mergern Acquisition', 'Gain On Sale Of Security', 'Operating Expense', 'Other Operating Expenses', 'Depreciation And Amortization In Income St

In [ ]:
# Check the exact column names (years) in each income statement
for name in banks:
    print(f"\n{name} income statement columns (years):")
    print(income_statements[name].columns.tolist())
    
    print(f"\n{name} - sample of key rows:")
    for field in ["Net Interest Income", "Interest Income", 
                  "Interest Expense", "Total Revenue", 
                  "Operating Expense", "Net Income"]:
        if field in income_statements[name].index:
            print(f"  {field}: {income_statements[name].loc[field].tolist()}")
        else:
            print(f"  {field}: NOT FOUND")

# Check balance sheet key rows
for name in banks:
    print(f"\n{name} - balance sheet key rows:")
    for field in ["Total Assets", "Common Stock Equity", 
                  "Stockholders Equity", "Tangible Book Value"]:
        if field in balance_sheets[name].index:
            print(f"  {field}: {balance_sheets[name].loc[field].tolist()}")
        else:
            print(f"  {field}: NOT FOUND")


Barclays income statement columns (years):
[Timestamp('2025-12-31 00:00:00'), Timestamp('2024-12-31 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2022-12-31 00:00:00'), Timestamp('2021-12-31 00:00:00')]

Barclays — sample of key rows:
  Net Interest Income: [14501000000.0, 12936000000.0, 12709000000.0, 10572000000.0, nan]
  Interest Income: [36189000000.0, 38326000000.0, 35075000000.0, 19096000000.0, nan]
  Interest Expense: [21688000000.0, 25390000000.0, 22366000000.0, 8524000000.0, nan]
  Total Revenue: [29140000000.0, 26230000000.0, 25384000000.0, 25021000000.0, nan]
  Operating Expense: [16849000000.0, 15964000000.0, 16079000000.0, 15077000000.0, nan]
  Net Income: [7172000000.0, 6307000000.0, 5259000000.0, 5928000000.0, nan]

BNP Paribas income statement columns (years):
[Timestamp('2025-12-31 00:00:00'), Timestamp('2024-12-31 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2022-12-31 00:00:00'), Timestamp('2021-12-31 00:00:00')]

BNP Paribas — sample of key r